# Tutorial: eQTL Analysis with JaxQTL and TensorQTL using `cellink`

This tutorial demonstrates how to perform eQTL analysis using external tools JaxQTL and TensorQTL through the `cellink` package. The `cellink` package provides a unified interface to these powerful QTL mapping tools, making it easier to perform comprehensive genetic analyses on single-cell datasets. These tools provide powerful statistical methods for detecting quantitative trait loci (QTLs) in genomic data, with JaxQTL offering fast GPU-accelerated analysis and TensorQTL providing comprehensive cis- and trans-QTL mapping capabilities.

This notebook assumes familiarity with single-cell data processing and basic statistical genetics concepts. The `cellink` package provides convenient wrapper functions that handle data preparation and formatting for these external tools. Install JaxQTL with `pip install 'cellink-tools[jaxqtl]'` (see its [repository](https://github.com/mancusolab/jaxqtl)) and TensorQTL with `pip install 'cellink-tools[tensorqtl]'`. For visualization of QTL calling results, please consider checking out the [Tutorial: Pseudobulk eQTL Analysis with `cellink`](https://cellink-docs.readthedocs.io/en/latest/tutorials/pseudobulk_eqtl.html).

## Environment Setup

We begin by importing necessary libraries and defining key parameters for our analysis. The `cellink` package provides wrapper functions for JaxQTL and TensorQTL that automatically handle data formatting and preparation.

In [1]:
import numpy as np
import scanpy as sc

from cellink.resources import get_dummy_onek1k
from cellink.tl.external import run_jaxqtl, run_tensorqtl

# Analysis parameters
n_gpcs = 20
n_epcs = 15
batch_e_pcs_n_top_genes = 2000
chrom = 22
cis_window = 500_000
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"
maf_threshold = 0.05  # test only variants with a minor allele frequency of at least 5%
min_gene_expr_pct = 0.1  # JaxQTL: test only genes expressed in at least 10% of donors

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Load and Prepare Data

We load the OneK1K dataset, which contains both genotype and single-cell expression data. Its gene annotations (Ensembl coordinates), which define genomic positions and cis-windows, are already included. (This is a subset of the full OneK1K dataset, which can be downloaded, and prepared using `get_onek1k()`)

In [2]:
# Load the dataset
dd = get_dummy_onek1k(verify_checksum=False)
print(f"Dataset shape: {dd.shape}")

dd.G.obsm["gPCs"] = dd.G.obsm["gPCs"][dd.G.obsm["gPCs"].columns[:n_gpcs]]

[2026-11-15 17:16:48,177] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.dd.h5 already exists
[2026-11-15 17:16:48,177] WARNING:root: No checksum provided, skipping verification
[2026-11-15 17:16:55,360] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)
Dataset shape: (300, 146939, 125913, 34830)



## Data Preprocessing

We filter the dataset to focus on a specific cell type and prepare the data for eQTL analysis.


In [3]:
dd.aggregate(obs=["donor_id", "sex", "age"], func="first", add_to_obs=True)

In [4]:
# Filter to specific cell type
dd = dd[..., dd.C.obs[celltype_key] == cell_type, :].copy()
print(f"After cell type filtering: {dd.shape}")

# Add donor-level metadata
dd.G.obs["donor_sex"] = dd.G.obs["sex"]
dd.G.obs["donor_age"] = dd.G.obs["age"]

# Generate random labels for demonstration (replace with real phenotypes)
dd.G.obs["donor_labels"] = np.random.randint(2, size=len(dd.G.obs))

# Keep the raw counts for JaxQTL's count models and log-normalize X for the Gaussian models (JaxQTL
# gaussian, TensorQTL). Normalizing before the chromosome filter computes the library sizes of X over all
# genes; JaxQTL's count-model offset uses the library sizes of the genes passed to it (chromosome 22 here).
dd.C.layers["counts"] = dd.C.X.copy()
sc.pp.normalize_total(dd.C, target_sum=1e4)
sc.pp.log1p(dd.C)

# Filter to specific chromosome for faster analysis
dd = dd.sel(G_var=dd.G.var.chrom == str(chrom), C_var=dd.C.var.chrom == str(chrom)).copy()
print(f"After chromosome {chrom} filtering: {dd.shape}")

After cell type filtering: (294, 146939, 4859, 34830)
After chromosome 22 filtering: (294, 136776, 4859, 888)


To speed up the computation we keep only the SNPs between 41.9 and 45.6 Mb. This region holds the two strongest CD8 naive T cell cis-eQTLs on chromosome 22 (SMDT1 and FAM118A).

In [5]:
dd = dd[:, (dd.G.var["pos"] > 41_900_000) & (dd.G.var["pos"] < 45_600_000), :, :].copy()
print(f"After SNP filtering: {dd.shape}")

After SNP filtering: (294, 17569, 4859, 888)


## JaxQTL Analysis

JaxQTL is a fast, GPU-accelerated tool for QTL mapping. It supports various statistical models and can handle large-scale genomic data efficiently. `run_jaxqtl` runs the `jaxqtl` subcommands `"cis"`, `"nominal"` and `"trans"` (`mode`); every `jaxqtl` option is a parameter of the same name. The count models (`nb`, `poisson`) get the raw counts summed per donor (`counts_layer="counts"`), the `gaussian` model the mean log-normalized expression per donor. Genes expressed in fewer than 10% of donors are skipped (`min_gene_expr_pct`): with only a few non-zero donors the per-variant tests are not valid.

### Running JaxQTL with Default Parameters
The basic `cis` mode performs permutation-based cis-QTL mapping: `pvalue_adj` is a Beta-approximated permutation p-value per gene (corrected for the variants tested for that gene, not across genes). This is the standard approach for identifying cis-eQTLs with appropriate multiple testing correction.

In [6]:
# Basic JaxQTL analysis
results_jaxqtl_basic = run_jaxqtl(
    dd,
    prefix="jaxqtl_basic",
    mode="cis",
    model="nb",  # Negative binomial model
    counts_layer="counts",  # raw counts for the count model
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    window=cis_window,
    additional_covariates=["gPCs"],
    run=True,
)
results_jaxqtl_basic

[2026-11-15 17:17:14,276] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 17:17:14,277] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:17:14,845] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:17:15,130] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_basic.bed/.bim/.fam (294 x 17569)
[2026-11-15 17:17:15,131] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_basic_phenotype.bed.gz --covar jaxqtl_basic_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --out jaxqtl_basic_donor_features_pcs.tsv


[2026-11-15 17:17:56 - INFO] Reading expression and selecting samples
[2026-11-15 17:18:01 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 17:18:01 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 17:18:01 - INFO] Using library sizes computed across all input genes before gene filtering
[2026-11-15 17:18:01 - INFO] Normalizing expression with library-size
[2026-11-15 17:18:02 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 17:18:10 - INFO] Computing 2 gene expression principal components for 294 samples
[2026-11-15 17:18:16 - INFO] Finished computing 2 gene expression principal components
[2026-11-15 17:18:16 - INFO] ExprPC1 proportion of variance explained: 0.0285634433
[2026-11-15 17:18:16 - INFO] ExprPC2 proportion of variance explained: 0.012185463
[2026-11-15 17:18:16 - INFO] Appending principal components to aligned covariates
[2026-11-15 17:18:16 - INFO] Writing results.
[2026-

[2026-11-15 17:18:18,182] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl cis --bfile jaxqtl_basic --pheno jaxqtl_basic_phenotype.bed.gz --covar jaxqtl_basic_donor_features_pcs.tsv --model nb --test score --nperm 1000 --window 500000 --normalize-covar --set-offset-from-libsize --min-gene-expr-pct 0.1 --maf 0.05 --out jaxqtl_basic


[2026-11-15 17:18:19 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 17:18:20 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 17:18:20 - INFO] Starting cis-scan.
[2026-11-15 17:30:21 - INFO] Finished cis-scan. Wrote results to jaxqtl_basic.cis.score.perm.parquet.gz.
[2026-11-15 17:30:21 - INFO] Finished! Thank you!


,phenotype_id,chrom,num_var,snp,a1,a0,pos,tss_distance,af,ma_count,...,beta,se,pvalue,pvalue_adj,adj_method,nb_alpha,negloglikelihood,model_converged,result_valid,failure_reason
0,ENSG00000183864,22,120,22_41924487_C_CA,CA,C,41924487,490992,0.079932,47,...,-0.420812,0.251400,9.415529e-02,7.143018e-01,BETA,2.380574e-02,246.399528,True,True,NaN
1,ENSG00000100412,22,368,22_41980960_G_A,A,G,41980960,533129,0.829932,100,...,-0.156994,0.094427,9.639476e-02,8.998758e-01,BETA,1.000000e-09,391.917205,True,True,NaN
2,ENSG00000100410,22,220,22_41950724_A_G,G,A,41950724,491006,0.102041,60,...,0.273233,0.124178,2.778287e-02,4.849918e-01,BETA,1.000000e-09,355.304105,True,True,NaN
3,ENSG00000100413,22,390,22_41994883_CTG_C,C,CTG,41994883,469083,0.416667,245,...,-0.316751,0.120112,8.360977e-03,3.455441e-01,BETA,1.000000e-09,228.873949,True,True,NaN
4,ENSG00000100417,22,468,22_41965078_C_CT,CT,C,41965078,388177,0.093537,55,...,-0.380658,0.160068,1.740195e-02,2.904894e-01,BETA,1.000000e-09,292.969823,True,True,NaN
5,ENSG00000100418,22,569,22_41967157_A_G,G,A,41967157,369128,0.071429,42,...,-0.471201,0.204637,2.130020e-02,5.040759e-01,BETA,1.000000e-09,298.631755,True,True,NaN
6,ENSG00000196419,22,751,22_42147079_G_GAAA,GAAA,G,42147079,525915,0.170068,100,...,-0.118313,0.051451,2.147484e-02,7.030522e-01,BETA,1.846484e-02,590.749139,True,True,NaN
7,ENSG00000100138,22,803,22_42139450_C_T,T,C,42139450,465516,0.073129,43,...,0.149796,0.064184,1.960359e-02,7.127364e-01,BETA,2.520874e-03,613.426437,True,True,NaN
8,ENSG00000167077,22,999,22_41988121_A_G,G,A,41988121,288617,0.829932,100,...,-0.398422,0.119492,8.551784e-04,9.444131e-02,BETA,1.000000e-09,331.923274,True,True,NaN
9,ENSG00000100147,22,1075,22_41924487_C_CA,CA,C,41924487,123807,0.079932,47,...,-1.126372,0.422598,7.690749e-03,4.869187e-01,BETA,1.000000e-09,134.056956,True,True,NaN


### Advanced JaxQTL Analysis with Custom Parameters
The `cis` mode with `acat=True` uses the Aggregated Cauchy Association Test to combine p-values across multiple variants, providing a powerful approach for detecting associations when there are multiple causal variants in a region. Its `pvalue_adj` is this gene-level ACAT p-value; it is not permutation-calibrated and not corrected across genes.

In [7]:
# Advanced JaxQTL analysis with more parameters
results_jaxqtl_advanced = run_jaxqtl(
    dd,
    prefix="jaxqtl_advanced",
    mode="cis",
    acat=True,  # gene-level ACAT p-values instead of permutations
    model="gaussian",  # Gaussian model
    window=1000000,  # 1Mb window
    test="wald",  # Wald test
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    additional_covariates=["gPCs"],
    expression_pcs=5,  # expression PCs added as covariates (jaxqtl compute-pcs)
    normalize_covar=True,
    verbose=True,
    run=True,
)
results_jaxqtl_advanced

[2026-11-15 17:30:23,034] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:30:23,035] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:30:24,193] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:30:24,452] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_advanced.bed/.bim/.fam (294 x 17569)
[2026-11-15 17:30:24,452] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_advanced_phenotype.bed.gz --covar jaxqtl_advanced_donor_features.tsv --num-pcs 5 --normalization library-size --transform log1p --verbose --out jaxqtl_advanced_donor_features_pcs.tsv


[2026-11-15 17:31:07 - INFO] Reading expression and selecting samples
[2026-11-15 17:31:12 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 17:31:13 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 17:31:13 - INFO] Using library sizes computed across all input genes before gene filtering
[2026-11-15 17:31:13 - INFO] Normalizing expression with library-size
[2026-11-15 17:31:14 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 17:31:15 - INFO] Computing 5 gene expression principal components for 294 samples
[2026-11-15 17:31:21 - INFO] Finished computing 5 gene expression principal components
[2026-11-15 17:31:21 - INFO] ExprPC1 proportion of variance explained: 0.013113318
[2026-11-15 17:31:21 - INFO] ExprPC2 proportion of variance explained: 0.0119646342
[2026-11-15 17:31:21 - INFO] ExprPC3 proportion of variance explained: 0.0116209946
[2026-11-15 17:31:21 - INFO] ExprPC4 proportion of

[2026-11-15 17:31:22,714] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl cis --bfile jaxqtl_advanced --pheno jaxqtl_advanced_phenotype.bed.gz --covar jaxqtl_advanced_donor_features_pcs.tsv --model gaussian --test wald --acat --window 1000000 --normalize-covar --min-gene-expr-pct 0.1 --maf 0.05 --verbose --out jaxqtl_advanced


[2026-11-15 17:31:24 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 17:31:24 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 17:31:24 - INFO] Starting cis-scan.
[2026-11-15 17:31:24 - WARNING] No cis-SNPs found for ENSG00000283633 over region 22:15601911-17615111. Skipping.
[2026-11-15 17:31:24 - WARNING] No cis-SNPs found for ENSG00000177663 over region 22:16084954-18115694. Skipping.
[2026-11-15 17:31:25 - WARNING] No cis-SNPs found for ENSG00000069998 over region 22:16137511-18165287. Skipping.
[2026-11-15 17:31:25 - WARNING] No cis-SNPs found for ENSG00000093072 over region 22:16178790-18258235. Skipping.
[2026-11-15 17:31:25 - WARNING] No cis-SNPs found for ENSG00000131100 over region 22:16592136-18628749. Skipping.
[2026-11-15 17:31:25 - WARNING] No cis-SNPs found for ENSG00000099968 over region 22:16628855-18730855. Skipping.
[2026-11-15 17:31:25 - WARNING] No cis-SNPs found for ENSG00000015475 over region 22:16734138-1

,phenotype_id,chrom,num_var,snp,a1,a0,pos,tss_distance,af,ma_count,beta,se,pvalue,pvalue_adj,adj_method,negloglikelihood,model_converged,result_valid,failure_reason
0,ENSG00000196236,22,79,22_41928618_T_G,G,T,41928618,1071540,0.197279,116,-0.020553,0.009779,3.653040e-02,4.560988e-01,ACAT,-296.777539,True,True,NaN
1,ENSG00000100387,22,226,22_41948907_A_G,G,A,41948907,997559,0.241497,142,-0.058236,0.023925,1.558694e-02,2.043689e-01,ACAT,-25.061770,True,True,NaN
2,ENSG00000100393,22,788,22_42173118_G_C,C,G,42173118,1080525,0.057823,34,0.035575,0.014143,1.248124e-02,9.066008e-01,ACAT,-346.146874,True,True,NaN
3,ENSG00000100395,22,854,22_41924949_CAAAAA_C,C,CAAAAA,41924949,719666,0.074830,44,0.026187,0.011192,2.003989e-02,4.079041e-01,ACAT,-398.553481,True,True,NaN
4,ENSG00000100401,22,969,22_42182590_T_C,C,T,42182590,937810,0.232993,137,0.016645,0.006488,1.085515e-02,5.521227e-01,ACAT,-401.434489,True,True,NaN
5,ENSG00000100403,22,1142,22_42247439_C_CA,CA,C,42247439,945913,0.316327,186,0.021548,0.007766,5.918915e-03,5.912461e-01,ACAT,-339.109707,True,True,NaN
6,ENSG00000183864,22,1411,22_42386396_A_AC,AC,A,42386396,952901,0.190476,112,0.028254,0.009489,3.175040e-03,7.439547e-01,ACAT,-325.204484,True,True,NaN
7,ENSG00000100412,22,1614,22_42474157_C_A,A,C,42474157,1026326,0.105442,62,0.074699,0.021319,5.380012e-04,8.935034e-01,ACAT,-170.590250,True,True,NaN
8,ENSG00000100410,22,1477,22_42458900_G_T,T,G,42458900,999182,0.069728,41,0.088981,0.022770,1.182894e-04,1.086121e-01,ACAT,-206.072557,True,True,NaN
9,ENSG00000100413,22,1626,22_42526579_C_CT,CT,C,42526579,1000779,0.214286,126,0.027676,0.008652,1.548185e-03,7.911980e-01,ACAT,-355.556457,True,True,NaN


### JaxQTL Trans-QTL Analysis
Trans-QTL analysis identifies associations between variants and distant genes, usually on other chromosomes (here all genes and variants are on chromosome 22, so the scan also covers cis pairs). These typically have smaller effect sizes than cis-QTLs and require larger sample sizes to detect.

In [8]:
# Trans-QTL analysis with JaxQTL
results_jaxqtl_trans = run_jaxqtl(
    dd,
    prefix="jaxqtl_trans",
    mode="trans",
    model="nb",
    counts_layer="counts",
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    additional_covariates=["gPCs"],
    run=True,
)
results_jaxqtl_trans

[2026-11-15 17:31:36,922] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 17:31:36,922] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:31:37,377] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:31:37,647] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_trans.bed/.bim/.fam (294 x 17569)
[2026-11-15 17:31:37,648] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl compute-pcs --pheno jaxqtl_trans_phenotype.bed.gz --covar jaxqtl_trans_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --out jaxqtl_trans_donor_features_pcs.tsv


[2026-11-15 17:31:39 - INFO] Reading expression and selecting samples
[2026-11-15 17:31:39 - INFO] Loaded 294 samples and 888 genes after sample selection
[2026-11-15 17:31:39 - INFO] Cohort alignment retained 294 samples; excluded 0 expression samples and 0 covariate samples
[2026-11-15 17:31:39 - INFO] Using library sizes computed across all input genes before gene filtering
[2026-11-15 17:31:39 - INFO] Normalizing expression with library-size
[2026-11-15 17:31:39 - INFO] Gene filters retained 504 of 888 genes
[2026-11-15 17:31:39 - INFO] Computing 2 gene expression principal components for 294 samples
[2026-11-15 17:31:40 - INFO] Finished computing 2 gene expression principal components
[2026-11-15 17:31:40 - INFO] ExprPC1 proportion of variance explained: 0.0285634433
[2026-11-15 17:31:40 - INFO] ExprPC2 proportion of variance explained: 0.012185463
[2026-11-15 17:31:40 - INFO] Appending principal components to aligned covariates
[2026-11-15 17:31:40 - INFO] Writing results.
[2026-

[2026-11-15 17:31:41,126] INFO:cellink.tl.external._jaxqtl: Running: jaxqtl trans --bfile jaxqtl_trans --pheno jaxqtl_trans_phenotype.bed.gz --covar jaxqtl_trans_donor_features_pcs.tsv --model nb --test score --window 500000 --normalize-covar --set-offset-from-libsize --min-gene-expr-pct 0.1 --maf 0.05 --out jaxqtl_trans


[2026-11-15 17:31:42 - INFO] Reading genotype, phenotype, and covariate data
[2026-11-15 17:31:43 - INFO] Finished reading and aligning genotype, phenotype, covariate data.
[2026-11-15 17:32:02 - INFO] Finished! Thank you!


,phenotype,snp,beta,se,pvalue,nb_alpha,model_converged,chrom,pos,a1,a0,af,ma_count
0,ENSG00000283633,22_41900699_T_C,0.463975,0.489760,0.343460,1.000000e-09,True,22,41900699,C,T,0.078231,46
1,ENSG00000283633,22_41901918_G_A,0.463975,0.489760,0.343460,1.000000e-09,True,22,41901918,A,G,0.078231,46
2,ENSG00000283633,22_41901964_T_C,0.463975,0.489760,0.343460,1.000000e-09,True,22,41901964,C,T,0.078231,46
3,ENSG00000283633,22_41902161_C_T,0.464294,0.489796,0.343164,1.000000e-09,True,22,41902161,T,C,0.076531,45
4,ENSG00000283633,22_41902597_C_T,0.463975,0.489760,0.343460,1.000000e-09,True,22,41902597,T,C,0.078231,46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3502417,ENSG00000079974,22_45599087_T_C,0.033081,0.091379,0.717340,1.000000e-09,True,22,45599087,C,T,0.658163,201
3502418,ENSG00000079974,22_45599476_T_C,-0.120370,0.129016,0.350827,1.000000e-09,True,22,45599476,C,T,0.171769,101
3502419,ENSG00000079974,22_45599746_C_T,-0.081641,0.179880,0.649927,1.000000e-09,True,22,45599746,T,C,0.088435,52
3502420,ENSG00000079974,22_45599912_C_T,-0.120370,0.129016,0.350827,1.000000e-09,True,22,45599912,T,C,0.171769,101


## TensorQTL Analysis

TensorQTL provides comprehensive QTL mapping capabilities with support for various analysis modes and statistical approaches. TensorQTL can be run in various modes: `["cis_nominal", "cis_independent", "cis", "trans", "cis_susie", "trans_susie"]`.

TensorQTL models continuous expression: `run_tensorqtl` averages the log-normalized expression per donor.

### Basic Cis-QTL Analysis with TensorQTL

The basic `cis` mode in TensorQTL performs permutation-based cis-QTL mapping, similar to JaxQTL but with PyTorch-based GPU acceleration.

In [9]:
# Basic cis-QTL mapping
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)
results_tensorqtl_cis

[2026-11-15 17:32:04,500] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:32:04,501] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:32:04,608] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:32:24,011] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:32:24,098] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:34:59,095] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:34:59,095] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:34:59,330] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:34:59,330] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:34:59,349] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:34:59,620] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:34:59,829] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:35:00] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_donor_featur

,phenotype_id,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,variant_id,start_distance,end_distance,ma_samples,ma_count,af,pval_nominal,slope,slope_se,pval_perm,pval_beta,qval,pval_nominal_threshold
0,ENSG00000273424,75,0.898986,4.67331,190.750,0.103490,22_41924487_C_CA,493552,493112,46,47,0.079932,0.053085,0.002216,0.001141,0.447655,0.448131,0.556254,0.001820
1,ENSG00000183864,120,0.924421,7.88688,250.820,0.278094,22_41903531_A_G,470036,456730,75,79,0.134354,0.261307,0.012606,0.011198,0.938106,0.932804,0.568547,0.001236
2,ENSG00000100412,368,0.946743,15.07440,226.697,0.026100,22_41992071_G_C,544240,462798,110,120,0.795918,0.015357,-0.044477,0.018233,0.347565,0.354315,0.556254,0.000727
3,ENSG00000100410,220,0.942702,11.04610,230.505,0.004057,22_41921891_CA_C,462173,453199,45,46,0.078231,0.001904,0.067120,0.021404,0.055894,0.053599,0.294638,0.000972
4,ENSG00000100413,390,0.926109,16.27810,230.376,0.068582,22_42008096_CCTTTCCTTTCCTTTCCTTTCCTTTG_C,482296,463490,44,47,0.079932,0.049043,-0.023407,0.011839,0.710529,0.713884,0.556254,0.000603
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,ENSG00000182257,176,0.842546,11.90410,217.833,0.102805,22_45569642_AAAGAAGAAGAAG_A,-479837,-484502,42,44,0.074830,0.069785,0.002534,0.001392,0.769323,0.779673,0.556254,0.000505
70,ENSG00000273289,176,0.757308,10.03380,213.045,0.077732,22_45574777_C_CT,-478317,-478783,42,44,0.074830,0.047407,0.001401,0.000703,0.636536,0.668935,0.556254,0.000327
71,ENSG00000241990,175,0.918048,15.38850,241.436,0.024737,22_45572077_C_T,-481629,-485133,197,251,0.573129,0.017767,-0.023408,0.009814,0.342766,0.359557,0.556254,0.000611
72,ENSG00000197182,175,0.852277,14.52970,244.476,0.161331,22_45596269_C_T,-457601,-517659,170,220,0.625850,0.141747,-0.006422,0.004358,0.948205,0.940978,0.568547,0.000440


### Nominal Cis-QTL Analysis
Nominal mode tests all variant-gene pairs within the specified window and returns comprehensive association statistics. This mode can also output top associations per gene, and significant pairs at the gene-level FDR when the output of a `cis` run is passed as `cis_output`.

In [10]:
# Nominal cis-QTL analysis (all variant-gene pairs)
results_tensorqtl_nominal = run_tensorqtl(
    dd,
    prefix="tensorqtl_nominal",
    mode="cis_nominal",
    window=cis_window,
    pval_threshold=1e-5,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    batch_size=20000,
    run=True,
)

# Results contain multiple outputs for nominal mode
cis_qtl_pairs, cis_qtl_signif_pairs, cis_qtl_top_assoc = results_tensorqtl_nominal
print(f"Total variant-gene pairs tested: {len(cis_qtl_pairs)}")
print(f"Pairs with p < 1e-5: {(cis_qtl_pairs['pval_nominal'] < 1e-5).sum()}")
cis_qtl_pairs

[2026-11-15 17:35:29,613] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:35:29,613] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:35:29,716] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:35:30,309] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:35:30,316] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_nominal.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:35:40,312] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:35:40,312] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:35:40,568] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:35:40,568] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:35:40,590] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:35:40,903] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:35:41,137] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:35:41] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_nominal_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_nominal_dono

,phenotype_id,variant_id,start_distance,end_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se
0,ENSG00000273424,22_41900699_T_C,469764,469324,0.078231,45,46,0.588905,-0.000614,0.001135
1,ENSG00000273424,22_41901918_G_A,470983,470543,0.078231,45,46,0.588905,-0.000614,0.001135
2,ENSG00000273424,22_41901964_T_C,471029,470589,0.078231,45,46,0.588905,-0.000614,0.001135
3,ENSG00000273424,22_41902161_C_T,471226,470786,0.076531,44,45,0.596524,-0.000607,0.001145
4,ENSG00000273424,22_41902597_C_T,471662,471222,0.078231,45,46,0.588905,-0.000614,0.001135
...,...,...,...,...,...,...,...,...,...,...
169425,ENSG00000231010,22_45599087_T_C,-456654,-459073,0.658163,161,201,0.223192,0.000371,0.000304
169426,ENSG00000231010,22_45599476_T_C,-456265,-458684,0.171769,89,101,0.249258,0.000452,0.000392
169427,ENSG00000231010,22_45599746_C_T,-455995,-458414,0.088435,48,52,0.589837,-0.000289,0.000535
169428,ENSG00000231010,22_45599912_C_T,-455829,-458248,0.171769,89,101,0.249258,0.000452,0.000392


### Cis-Independent QTL Analysis
The `cis_independent` mode identifies multiple independent association signals for each gene by performing conditional analysis. This is crucial for understanding the genetic architecture when multiple causal variants affect the same gene.

In [11]:
# First run basic cis mode to get top associations
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)

# Now identify independent signals conditioning on the top hits
results_tensorqtl_independent = run_tensorqtl(
    dd,
    prefix="tensorqtl_independent",
    mode="cis_independent",
    window=cis_window,
    cis_output="tensorqtl_cis.cis_qtl.txt.gz",  # Use output from cis mode
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=True,
)

print(f"Independent signals identified: {len(results_tensorqtl_independent)}")
results_tensorqtl_independent

[2026-11-15 17:35:45,866] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:35:45,866] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:35:45,976] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:35:48,738] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:35:48,746] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:35:59,360] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:35:59,360] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:35:59,615] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:35:59,615] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:35:59,637] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:35:59,928] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:36:00,162] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:36:00] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_donor_featur

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:36:25,219] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:36:25,219] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:36:25,466] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:25,467] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:25,487] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:36:25,768] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:36:25,993] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:36:26] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_independent_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_independ

,phenotype_id,num_var,beta_shape1,beta_shape2,true_df,pval_true_df,variant_id,start_distance,end_distance,ma_samples,ma_count,af,pval_nominal,slope,slope_se,pval_perm,pval_beta,rank
0,ENSG00000183172,1741,1.035910,70.3288,214.451,2.966160e-17,22_42120602_TAA_T,40910,36318,140,164,0.278912,2.972700e-21,-0.356959,0.034590,0.000100,6.102770e-16,1
1,ENSG00000270083,2361,0.698307,15.9322,114.073,1.547390e-11,22_42065183_T_C,-24448,-24845,28,31,0.052721,4.245010e-25,0.245477,0.021371,0.001000,2.135400e-07,1
2,ENSG00000270083,2361,0.706118,16.5719,116.360,1.652140e-09,22_42116188_C_T,26557,26160,29,31,0.052721,5.176360e-20,-0.218919,0.022031,0.007099,4.991910e-06,2
3,ENSG00000100376,3147,0.977807,188.6620,212.711,6.968180e-24,22_45393863_T_G,84894,51908,76,82,0.139456,8.797530e-30,0.315746,0.024576,0.000100,3.856590e-21,1


### Trans-QTL Analysis with TensorQTL
Trans-QTL mapping with TensorQTL tests associations between variants and distant genes. Due to the large number of tests, stricter p-value thresholds are typically required.

In [12]:
# Trans-QTL mapping
results_tensorqtl_trans = run_tensorqtl(
    dd,
    prefix="tensorqtl_trans",
    mode="trans",
    pval_threshold=1e-6,  # Stricter threshold for trans
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    batch_size=10000,  # Smaller batches for trans analysis
    return_r2=True,  # Include R² statistics
    run=True,
)
results_tensorqtl_trans

[2026-11-15 17:36:28,839] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:36:28,839] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:36:28,948] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:36:29,513] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:36:29,536] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_trans.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:36:39,818] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:36:39,818] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:36:40,063] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:40,063] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:40,083] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:36:40,367] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:36:40,591] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:36:40] Running TensorQTL v1.0.10: trans-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_trans_phenotype.bed.gz)
  * reading covariates (tensorqtl_trans_donor_features.tsv)
  * loading genotypes
  * p-value threshold: 1

,variant_id,phenotype_id,pval,b,b_se,r2,af
0,22_41900699_T_C,ENSG00000211660,3.695876e-07,0.008235,0.001580,0.091775,0.078231
2,22_41901918_G_A,ENSG00000211660,3.695876e-07,0.008235,0.001580,0.091775,0.078231
4,22_41901964_T_C,ENSG00000211660,3.695876e-07,0.008235,0.001580,0.091775,0.078231
6,22_41902161_C_T,ENSG00000211660,3.801149e-07,0.008299,0.001594,0.091591,0.076531
8,22_41902597_C_T,ENSG00000211660,3.695876e-07,0.008235,0.001580,0.091775,0.078231
...,...,...,...,...,...,...,...
1236,22_45537816_C_T,ENSG00000100079,4.503739e-07,0.029299,0.005664,0.090483,0.088435
1237,22_45547561_A_T,ENSG00000167037,4.917053e-07,0.090950,0.017643,0.089909,0.078231
1238,22_45574777_C_CT,ENSG00000211666,2.480030e-07,0.006337,0.001197,0.094377,0.074830
1239,22_45574777_C_CT,ENSG00000128311,6.506957e-07,0.015886,0.003117,0.088077,0.074830


### SuSiE Finemapping with TensorQTL
`SuSiE` (Sum of Single Effects) is a powerful finemapping method that identifies credible sets of causal variants for each independent association signal. It accounts for LD structure and provides posterior probabilities for each variant being causal.
#### Cis-SuSiE Finemapping
Cis-SuSiE performs finemapping on the top cis-QTL signals to identify credible sets of potentially causal variants. This requires the output from a previous cis-QTL run.

In [13]:
# First, run standard cis-QTL analysis if not already done
results_tensorqtl_cis = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis_for_susie",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    permutations=10000,
    run=True,
)

# Perform SuSiE finemapping on cis-QTL signals
results_tensorqtl_cis_susie = run_tensorqtl(
    dd,
    prefix="tensorqtl_cis_susie",
    mode="cis_susie",
    window=cis_window,
    cis_output="tensorqtl_cis_for_susie.cis_qtl.txt.gz",  # Input from cis mode
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    max_effects=10,  # Maximum number of causal effects per gene
    run=True,
)

# Results contain SuSiE model objects and summary statistics
susie_models, susie_summary = results_tensorqtl_cis_susie

print(f"Genes with finemapping results: {len(susie_models)}")
print("\nSummary statistics columns:")
print(susie_summary.columns.tolist())
print(f"\nCredible sets identified: {susie_summary.groupby(['phenotype_id', 'cs_id']).ngroups}")

# Examine credible sets for a specific gene
if len(susie_summary) > 0:
    example_gene = susie_summary["phenotype_id"].iloc[0]
    gene_credible_sets = susie_summary[susie_summary["phenotype_id"] == example_gene]
    print(f"\nCredible sets for {example_gene}:")
    print(gene_credible_sets)

[2026-11-15 17:36:42,943] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:36:42,943] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:36:43,053] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:36:45,756] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:36:45,765] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cis_for_susie.pgen/.pvar/.psam (294 x 17569)


site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:36:55,472] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:36:55,472] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:36:55,702] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:55,702] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:36:55,721] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:36:55,991] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:36:56,198] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:36:56] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_for_susie_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_fo

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-11-15 17:37:20,727] INFO:rpy2.situation: cffi mode is CFFI_MODE.ANY
[2026-11-15 17:37:20,727] INFO:rpy2.situation: R home found: /usr/lib/R
[2026-11-15 17:37:20,970] INFO:rpy2.situation: R library path: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:37:20,970] INFO:rpy2.situation: LD_LIBRARY_PATH: WARNING: ignoring environment value of R_HOME

[2026-11-15 17:37:20,990] INFO:rpy2.rinterface_lib.embedded: Default options to initialize R: rpy2, --quiet, --no-save
[2026-11-15 17:37:21,274] INFO:rpy2.rinterface: Environment variable "R_HOME" redefined by R and overriding existing variable. Current: "/usr/lib/R", R: "/usr/lib/R"
[2026-11-15 17:37:21,502] INFO:rpy2.rinterface: R is already initialized. No need to initialize.
[Nov 15 17:37:21] Running TensorQTL v1.0.10: cis-QTL mapping
  * WARNING: using CPU!
  * reading phenotypes (tensorqtl_cis_susie_phenotype.bed.gz)
  * cis-window detected as [start - 500,000, end + 500,000]
  * reading covariates (tensorqtl_cis_susie_

## Advanced Usage: Dry Run and Command Generation

Both JaxQTL and TensorQTL support generating commands without execution, which is useful for debugging or running on compute clusters. This is controlled by the `run` argument.

### JaxQTL Command Generation

In [14]:
# Generate JaxQTL command without running
jaxqtl_command = run_jaxqtl(
    dd,
    prefix="jaxqtl_cluster",
    mode="cis",
    additional_covariates=["gPCs"],
    counts_layer="counts",  # dd.C.X is log-normalized by now
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    run=False,  # Don't execute, just return command
)

print("JaxQTL command:")
print(jaxqtl_command)

[2026-11-15 17:37:24,368] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 17:37:24,368] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:37:24,829] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:37:25,102] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_cluster.bed/.bim/.fam (294 x 17569)
JaxQTL command:
jaxqtl compute-pcs --pheno jaxqtl_cluster_phenotype.bed.gz --covar jaxqtl_cluster_donor_features.tsv --num-pcs 2 --normalization library-size --transform log1p --out jaxqtl_cluster_donor_features_pcs.tsv && jaxqtl cis --bfile jaxqtl_cluster --pheno jaxqtl_cluster_phenotype.bed.gz --covar jaxqtl_cluster_donor_features_pcs.tsv --model nb --test score --nperm 1000 --window 500000 --normalize-covar --set-offset-from-libsize --min-gene-expr-pct 0.1 --maf 0.05 --out 

### TensorQTL Command Generation

In [15]:
# Generate TensorQTL command without running
tensorqtl_command = run_tensorqtl(
    dd,
    prefix="tensorqtl_cluster",
    mode="cis",
    window=cis_window,
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=False,  # Don't execute, just return command
)

print("\nTensorQTL command:")
print(tensorqtl_command)

[2026-11-15 17:37:25,114] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:37:25,115] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:37:25,223] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:37:25,805] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:37:25,821] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cluster.pgen/.pvar/.psam (294 x 17569)

TensorQTL command:
python -m cellink.tl.external._tensorqtl_main tensorqtl_cluster tensorqtl_cluster_phenotype.bed.gz tensorqtl_cluster --covariates tensorqtl_cluster_donor_features.tsv --mode cis --permutations 10000 --window 500000 --pval_threshold 1e-05 --maf_threshold 0.05 --maf_threshold_interaction 0.05 --batch_size 20000 --warn_monomorphic --max_effects 10 --fdr 0.05


#### Saving Commands to Files
You can also save the generated commands to files for batch submission to compute clusters:

In [16]:
# Save JaxQTL command to file
jaxqtl_command = run_jaxqtl(
    dd,
    prefix="jaxqtl_cluster",
    mode="cis",
    acat=True,
    model="nb",
    window=cis_window,
    additional_covariates=["gPCs"],
    counts_layer="counts",  # dd.C.X is log-normalized by now
    maf=maf_threshold,
    min_gene_expr_pct=min_gene_expr_pct,
    run=False,
    save_cmd_file="jaxqtl_job.sh",
)

[2026-11-15 17:37:25,834] INFO:cellink._core.donordata: Aggregated counts to PB
[2026-11-15 17:37:25,835] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:37:26,367] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:37:26,608] INFO:cellink.io._export: Wrote PLINK 1 binary files jaxqtl_cluster.bed/.bim/.fam (294 x 17569)


In [17]:
# Save TensorQTL command to file
tensorqtl_command = run_tensorqtl(
    dd,
    prefix="tensorqtl_cluster",
    mode="cis_susie",
    cis_output="previous_cis_results.txt.gz",
    additional_covariates=["gPCs"],
    maf_threshold=maf_threshold,
    run=False,
    save_cmd_file="tensorqtl_susie_job.sh",
)

[2026-11-15 17:37:26,629] INFO:cellink._core.donordata: Aggregated X to PB
[2026-11-15 17:37:26,630] INFO:cellink._core.donordata: Observation found for 294 donors.
[2026-11-15 17:37:27,151] INFO:cellink.tl.external._tensorqtl: Performing z-normalization of age.
[2026-11-15 17:37:27,723] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.
[2026-11-15 17:37:27,728] INFO:cellink.io._export: Wrote PLINK 2 binary files tensorqtl_cluster.pgen/.pvar/.psam (294 x 17569)
